# 02 - Nowcasting: Detecting Flares as They Happen

**Nowcasting** = spotting a flare *right now* in the live signal (versus *forecasting* the future, which is notebook 03). This part uses **no machine learning** - it's a simple rule: "if the brightness jumps far enough above the quiet background, that's a flare."

Here you'll run that detector on real SoLEXS data, check it against the saved catalog, and see how a sensitivity knob changes the results.

**Needs:** processed SoLEXS parquet in `data/processed/` (`python scripts/process_all.py --rebuild`).

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

from pipeline.detector import detect_solexs_flares, events_to_dataframe
from pipeline.catalog import catalog_summary, load_master_catalog

DATA = Path('../data')
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

## 1. Run the detector on the first 10 SoLEXS days

The detector scans each day's light curve and returns a list of **flare events**. Each event records when the flare **started**, **peaked**, and **ended**, how bright the peak was (counts/s), and its **class** (A/B/C/M/X). We use 10 days here to keep it quick; the table shows the first few flares found.

In [ ]:
sxr_files = sorted(DATA.glob('processed/*_solexs.parquet'))[:10]
all_events = []
for f in sxr_files:
    sx = pd.read_parquet(f)
    all_events.extend(detect_solexs_flares(sx, cfg))

detected = events_to_dataframe(all_events)
print(f'Days scanned: {len(sxr_files)}  |  flares detected: {len(detected)}')
print(catalog_summary(detected))
detected.head()

## 2. Sanity-check against the saved catalog

The full catalog was built the same way over *all* days. To check this run is consistent, we filter the catalog to the **same 10 days** and compare the counts, class by class. They should match closely (small differences are fine if the config changed).

In [ ]:
cat = load_master_catalog('../data/processed/master_catalog.parquet')
# Real apples-to-apples: filter the saved catalog to the SAME 10 days we just scanned.
days = {f.name[:8] for f in sxr_files}
cat_days = cat[cat['peak_time'].dt.strftime('%Y%m%d').isin(days)] if not cat.empty else cat
print(f'Same {len(sxr_files)} days  ->  detector now: {len(detected)}  |  in saved catalog: {len(cat_days)}')
if not detected.empty:
    here = detected['flare_class'].str[0].value_counts()
    there = cat_days['flare_class'].str[0].value_counts() if not cat_days.empty else pd.Series(dtype=int)
    print(pd.DataFrame({'detected_now': here, 'in_catalog': there}).fillna(0).astype(int))

## 3. How sensitive is the detector? (`sigma_threshold`)

The main knob is `sigma_threshold`: how many times above the background **noise** a bump must rise to count as a flare. **Lower** = more sensitive (catches small flares, but more false alarms from noise); **higher** = stricter (only strong, obvious flares). This cell re-runs detection on one day at several settings and plots how the flare count changes - handy for choosing a good value.

In [ ]:
if sxr_files:
    sx = pd.read_parquet(sxr_files[0])
    rows = []
    for sigma in [2.0, 2.5, 3.0, 3.5, 4.0]:
        test_cfg = {**cfg, 'nowcasting': {**cfg['nowcasting'], 'sigma_threshold': sigma}}
        rows.append({'sigma': sigma, 'n_detected': len(detect_solexs_flares(sx, test_cfg))})
    sweep = pd.DataFrame(rows); print(sweep)
    plt.figure(figsize=(6, 3)); plt.plot(sweep['sigma'], sweep['n_detected'], 'o-')
    plt.xlabel('sigma_threshold'); plt.ylabel('flares detected')
    plt.title('Detection sensitivity (1 SoLEXS day)'); plt.tight_layout(); plt.show()

## 4. Look at one flare up close

Numbers are abstract, so here we plot the single strongest flare from the days we scanned, zoomed to +/-1 hour around its peak. The dashed orange line is the background; the vertical lines mark the detector's **start** (green), **peak** (red), and **end** (purple). This is exactly what a "detection" looks like on a real light curve.

In [ ]:
if not detected.empty:
    ev = detected.loc[detected['peak_flux'].idxmax()]
    print(f"Strongest: {ev['flare_class']} at {ev['peak_time']}")
    day = pd.Timestamp(ev['peak_time']).strftime('%Y%m%d')
    fpath = DATA / f'processed/{day}_solexs.parquet'
    if fpath.exists():
        sx = pd.read_parquet(fpath)
        t0 = pd.Timestamp(ev['peak_time']) - pd.Timedelta(hours=1)
        t1 = pd.Timestamp(ev['peak_time']) + pd.Timedelta(hours=1)
        w = sx.loc[t0:t1]
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(w.index, w['flux_long'], lw=0.7, label='flux_long')
        if 'bg_long' in w:
            ax.plot(w.index, w['bg_long'], '--', color='orange', label='background')
        ax.axvline(ev['start_time'], color='green', ls=':', label='start')
        ax.axvline(ev['peak_time'], color='red', label=f"peak ({ev['flare_class']})")
        if pd.notnull(ev['end_time']):
            ax.axvline(ev['end_time'], color='purple', ls=':', label='end')
        ax.set_yscale('log'); ax.set_ylabel('counts/s')
        ax.set_title(f"SoLEXS flare {ev['flare_class']}"); ax.legend()
        plt.tight_layout(); plt.show()